# 🧠 NeuroShield AI — Real Audio Version



## Step 1 — Install

In [1]:
%%capture
!pip install streamlit>=1.33.0 deepface tf-keras transformers torch librosa soundfile pyngrok pillow opencv-python-headless sentencepiece

## Step 2 — ngrok token


In [2]:
NGROK_TOKEN = "3DAJQhjVg63Tbf9OENqkRdpZkfK_7i17yJWjRQALRHCiULLWP"
from pyngrok import ngrok, conf
conf.get_default().auth_token = NGROK_TOKEN
print("✅ ngrok ready")

✅ ngrok ready


In [3]:
import builtins
import io


def _open_with_utf8_default(*args, **kwargs):
    mode = args[1] if len(args) > 1 else kwargs.get("mode", "r")
    if "encoding" not in kwargs and any(flag in mode for flag in ("w", "a", "x", "+")):
        kwargs["encoding"] = "utf-8"
    return io.open(*args, **kwargs)


builtins.open = _open_with_utf8_default
globals()["open"] = _open_with_utf8_default
print("UTF-8 text-write default enabled")

UTF-8 text-write default enabled


## Step 3 — Write the app

In [4]:
app_code = 'import streamlit as st\nimport numpy as np\nimport cv2\nimport time\nimport io\nimport warnings\nwarnings.filterwarnings("ignore")\n\nst.set_page_config(\n    page_title="NeuroShield AI",\n    page_icon="🧠",\n    layout="wide",\n    initial_sidebar_state="expanded"\n)\n\nst.markdown("""\n<style>\n@import url(\'https://fonts.googleapis.com/css2?family=JetBrains+Mono:wght@400;600;700&family=Inter:wght@300;400;500;600&display=swap\');\n\nhtml, body, [class*="css"] { font-family: \'Inter\', sans-serif; background: #070b14; color: #cdd6f4; }\n.stApp { background: #070b14; }\n\n[data-testid="stSidebar"] { background: #0d1117; border-right: 1px solid #1e2d40; }\n[data-testid="stSidebar"] * { color: #8b9ab3 !important; font-size: 0.82rem; }\n[data-testid="stSidebar"] h3 { color: #7aa2f7 !important; font-size: 0.75rem !important; letter-spacing: 1.5px; text-transform: uppercase; }\n\n.ns-header { background: #0d1117; border: 1px solid #1e2d40; border-radius: 16px; padding: 1.8rem 2.4rem; margin-bottom: 1.6rem; }\n.ns-logo { font-family: \'JetBrains Mono\', monospace; font-size: 1.8rem; font-weight: 700; background: linear-gradient(120deg, #7aa2f7, #7dcfff, #bb9af7); -webkit-background-clip: text; -webkit-text-fill-color: transparent; }\n.ns-tagline { font-size: 0.82rem; color: #4a5568; margin-top: 0.3rem; }\n.ns-badges { display: flex; gap: 8px; margin-top: 0.9rem; flex-wrap: wrap; }\n.badge { font-family: \'JetBrains Mono\', monospace; font-size: 0.65rem; font-weight: 600; padding: 3px 10px; border-radius: 4px; letter-spacing: 0.5px; text-transform: uppercase; }\n.badge-blue   { background: rgba(122,162,247,0.12); color: #7aa2f7; border: 1px solid rgba(122,162,247,0.25); }\n.badge-green  { background: rgba(158,206,106,0.12); color: #9ece6a; border: 1px solid rgba(158,206,106,0.25); }\n.badge-purple { background: rgba(187,154,247,0.12); color: #bb9af7; border: 1px solid rgba(187,154,247,0.25); }\n.badge-teal   { background: rgba(125,207,255,0.12); color: #7dcfff; border: 1px solid rgba(125,207,255,0.25); }\n.badge-red    { background: rgba(247,118,142,0.12); color: #f7768e; border: 1px solid rgba(247,118,142,0.25); }\n\n.card { background: #0d1117; border: 1px solid #1e2d40; border-radius: 12px; padding: 1.3rem; margin-bottom: 1rem; }\n.card-blue   { border-left: 3px solid #7aa2f7; }\n.card-green  { border-left: 3px solid #9ece6a; }\n.card-purple { border-left: 3px solid #bb9af7; }\n\n.clabel { font-family: \'JetBrains Mono\', monospace; font-size: 0.65rem; font-weight: 600; letter-spacing: 1.8px; text-transform: uppercase; color: #4a5568; margin-bottom: 0.8rem; }\n.metric-big { font-family: \'JetBrains Mono\', monospace; font-size: 2.2rem; font-weight: 700; line-height: 1; }\n.metric-sub { font-size: 0.75rem; color: #4a5568; margin-top: 0.3rem; }\n\n.pbar-wrap   { background: #131b2e; border-radius: 4px; height: 7px; overflow: hidden; margin: 5px 0; }\n.pbar        { height: 100%; border-radius: 4px; }\n.pbar-blue   { background: linear-gradient(90deg, #1f4f8f, #7aa2f7); }\n.pbar-green  { background: linear-gradient(90deg, #2a5e1e, #9ece6a); }\n.pbar-purple { background: linear-gradient(90deg, #4a2c7a, #bb9af7); }\n.pbar-red    { background: linear-gradient(90deg, #7a1f2e, #f7768e); }\n.pbar-amber  { background: linear-gradient(90deg, #6b4e10, #e0af68); }\n\n.chip { display: inline-flex; align-items: center; font-size: 0.7rem; font-weight: 600; padding: 3px 9px; border-radius: 4px; margin: 2px; font-family: \'JetBrains Mono\', monospace; }\n.chip-angry   { background: rgba(247,118,142,0.1); color: #f7768e; border: 1px solid rgba(247,118,142,0.25); }\n.chip-happy   { background: rgba(158,206,106,0.1); color: #9ece6a; border: 1px solid rgba(158,206,106,0.25); }\n.chip-sad     { background: rgba(125,207,255,0.1); color: #7dcfff; border: 1px solid rgba(125,207,255,0.25); }\n.chip-fear    { background: rgba(224,175,104,0.1); color: #e0af68; border: 1px solid rgba(224,175,104,0.25); }\n.chip-neutral { background: rgba(139,154,179,0.1); color: #8b9ab3; border: 1px solid rgba(139,154,179,0.25); }\n.chip-disgust { background: rgba(187,154,247,0.1); color: #bb9af7; border: 1px solid rgba(187,154,247,0.25); }\n.chip-surprise{ background: rgba(255,158,100,0.1); color: #ff9e64; border: 1px solid rgba(255,158,100,0.25); }\n\n.risk-box { border-radius: 14px; padding: 2rem 2.2rem; margin-top: 0.5rem; border: 2px solid; }\n.risk-low    { background: rgba(158,206,106,0.04); border-color: rgba(158,206,106,0.35); }\n.risk-medium { background: rgba(224,175,104,0.04); border-color: rgba(224,175,104,0.35); }\n.risk-high   { background: rgba(247,118,142,0.04); border-color: rgba(247,118,142,0.35); }\n.risk-val    { font-family: \'JetBrains Mono\', monospace; font-size: 2.8rem; font-weight: 700; line-height: 1; }\n.rv-low { color: #9ece6a; } .rv-medium { color: #e0af68; } .rv-high { color: #f7768e; }\n\n.xai-title { font-family: \'JetBrains Mono\', monospace; font-size: 0.65rem; font-weight: 700; letter-spacing: 2px; text-transform: uppercase; color: #4a5568; margin-bottom: 0.9rem; }\n.xai-row   { display: flex; align-items: flex-start; gap: 10px; margin-bottom: 0.75rem; font-size: 0.82rem; color: #8b9ab3; line-height: 1.55; }\n.xai-dot   { width: 7px; height: 7px; border-radius: 50%; margin-top: 5px; flex-shrink: 0; }\n.dot-face  { background: #7aa2f7; box-shadow: 0 0 6px rgba(122,162,247,0.5); }\n.dot-voice { background: #9ece6a; box-shadow: 0 0 6px rgba(158,206,106,0.5); }\n.dot-text  { background: #bb9af7; box-shadow: 0 0 6px rgba(187,154,247,0.5); }\n\n.formula-box { background: #131b2e; border: 1px solid #1e2d40; border-radius: 8px; padding: 0.9rem 1.1rem; font-family: \'JetBrains Mono\', monospace; font-size: 0.75rem; color: #7aa2f7; margin-top: 0.8rem; line-height: 1.9; }\n.fl-badge    { background: #0a0f1a; border: 1px solid #1e2d40; border-radius: 8px; padding: 0.9rem 1.1rem; font-family: \'JetBrains Mono\', monospace; font-size: 0.72rem; color: #4a5568; margin-top: 1rem; line-height: 1.9; }\n\n.feat-row { display: flex; justify-content: space-between; font-size: 0.76rem; margin-bottom: 5px; }\n.feat-key { color: #4a5568; }\n.feat-val { color: #cdd6f4; font-family: \'JetBrains Mono\', monospace; }\n\n.live-badge { display: inline-block; background: rgba(158,206,106,0.15); border: 1px solid rgba(158,206,106,0.4); color: #9ece6a; font-size: 0.65rem; font-family: \'JetBrains Mono\', monospace; padding: 2px 8px; border-radius: 4px; margin-left: 6px; animation: pulse 2s infinite; }\n@keyframes pulse { 0%,100%{opacity:1} 50%{opacity:0.5} }\n\ndiv.stButton > button { background: linear-gradient(135deg, #1e3a5f, #2a4a7f); color: #7aa2f7; border: 1px solid rgba(122,162,247,0.4); border-radius: 8px; padding: 0.55rem 1.2rem; font-family: \'JetBrains Mono\', monospace; font-size: 0.78rem; font-weight: 600; width: 100%; }\ntextarea, input[type="text"] { background: #0d1117 !important; border: 1px solid #1e2d40 !important; border-radius: 8px !important; color: #cdd6f4 !important; font-family: \'Inter\', sans-serif !important; }\n#MainMenu, footer, header { visibility: hidden; }\n.stSpinner > div { border-top-color: #7aa2f7 !important; }\n</style>\n""", unsafe_allow_html=True)\n\n\n# ══════════════════════════════════════════════════════════════\n# REAL VOICE ANALYSIS — actual Librosa pipeline\n# ══════════════════════════════════════════════════════════════\ndef analyze_voice_real(audio_bytes: bytes) -> dict:\n    """\n    REAL Librosa feature extraction — NOT simulated.\n    Extracts MFCCs, ZCR, pitch, spectral centroid, RMS energy\n    from actual recorded audio bytes.\n    """\n    try:\n        import librosa\n        import soundfile as sf\n\n        audio_buffer = io.BytesIO(audio_bytes)\n\n        # Load audio — soundfile handles wav/flac/ogg natively\n        try:\n            y, sr = sf.read(audio_buffer)\n            if len(y.shape) > 1:\n                y = np.mean(y, axis=1)   # stereo → mono\n            y = y.astype(np.float32)\n        except Exception:\n            audio_buffer.seek(0)\n            y, sr = librosa.load(audio_buffer, sr=None, mono=True)\n\n        # Resample to 16 kHz — standard for speech\n        TARGET_SR = 16000\n        if sr != TARGET_SR:\n            y = librosa.resample(y, orig_sr=sr, target_sr=TARGET_SR)\n            sr = TARGET_SR\n\n        if len(y) < sr * 0.3:\n            return {"ok": False, "error": "Too short — speak for at least 1 second"}\n\n        # ── MFCC extraction ───────────────────────────────────\n        N_FFT, HOP, N_MFCC = 512, 160, 40\n        mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=N_MFCC,\n                                     n_fft=N_FFT, hop_length=HOP)\n        mfcc_mean = np.mean(mfcc, axis=1)\n\n        # Delta MFCCs (speech dynamics)\n        d_mfcc  = librosa.feature.delta(mfcc)\n        d2_mfcc = librosa.feature.delta(mfcc, order=2)\n\n        # ── Spectral features ─────────────────────────────────\n        zcr = float(np.mean(\n            librosa.feature.zero_crossing_rate(y, hop_length=HOP)\n        ))\n        spectral_centroid = float(np.mean(\n            librosa.feature.spectral_centroid(y=y, sr=sr,\n                                              n_fft=N_FFT, hop_length=HOP)\n        ))\n        rms = float(np.mean(\n            librosa.feature.rms(y=y, hop_length=HOP)\n        ))\n        rolloff = float(np.mean(\n            librosa.feature.spectral_rolloff(y=y, sr=sr,\n                                             n_fft=N_FFT, hop_length=HOP,\n                                             roll_percent=0.85)\n        ))\n\n        # ── Pitch (F0) via YIN algorithm ──────────────────────\n        try:\n            f0 = librosa.yin(y, fmin=60, fmax=400, sr=sr, hop_length=HOP)\n            voiced = f0[f0 > 60]\n            pitch_mean = float(np.mean(voiced)) if len(voiced) > 5 else 150.0\n            pitch_std  = float(np.std(voiced))  if len(voiced) > 5 else 25.0\n        except Exception:\n            pitch_mean, pitch_std = 150.0, 25.0\n\n        # ── Spectral bandwidth ────────────────────────────────\n        bandwidth = float(np.mean(\n            librosa.feature.spectral_bandwidth(y=y, sr=sr,\n                                               n_fft=N_FFT, hop_length=HOP)\n        ))\n\n        # ── RISK SCORE computation ────────────────────────────\n        # Normalise each feature to [0,1] using speech stress ranges\n        # (Schuller et al. 2013, El Ayadi et al. 2011)\n        zcr_n       = min(zcr / 0.15,          1.0)\n        rms_n       = min(rms / 0.12,          1.0)\n        pitch_n     = min(max(pitch_mean - 80, 0) / 250.0, 1.0)\n        pitch_var_n = min(pitch_std / 70.0,    1.0)\n        centroid_n  = min(spectral_centroid / 3000.0, 1.0)\n\n        risk = (0.25 * zcr_n +\n                0.20 * rms_n +\n                0.20 * pitch_n +\n                0.20 * pitch_var_n +\n                0.15 * centroid_n)\n        risk = float(np.clip(risk, 0.0, 1.0))\n\n        return {\n            "ok":                 True,\n            "risk_score":         round(risk, 4),\n            "pitch_mean":         round(pitch_mean, 1),\n            "pitch_std":          round(pitch_std, 1),\n            "zcr":                round(zcr, 5),\n            "spectral_centroid":  round(spectral_centroid, 1),\n            "rms_energy":         round(rms, 5),\n            "rolloff":            round(rolloff, 1),\n            "bandwidth":          round(bandwidth, 1),\n            "mfcc1":              round(float(mfcc_mean[0]), 2),\n            "mfcc2":              round(float(mfcc_mean[1]), 2),\n            "mfcc3":              round(float(mfcc_mean[2]), 2),\n            "duration_sec":       round(len(y) / sr, 2),\n        }\n    except Exception as e:\n        return {"ok": False, "error": str(e)}\n\n\n# ══════════════════════════════════════════════════════════════\n# MODELS (cached)\n# ══════════════════════════════════════════════════════════════\n@st.cache_resource(show_spinner=False)\ndef load_sentiment():\n    from transformers import pipeline\n    return pipeline("sentiment-analysis",\n                    model="distilbert-base-uncased-finetuned-sst-2-english")\n\n@st.cache_resource(show_spinner=False)\ndef load_emotion():\n    from transformers import pipeline\n    return pipeline("text-classification",\n                    model="j-hartmann/emotion-english-distilroberta-base",\n                    top_k=None)\n\n\n# ══════════════════════════════════════════════════════════════\n# FACE ANALYSIS\n# ══════════════════════════════════════════════════════════════\ndef analyze_face(img_bytes):\n    try:\n        from deepface import DeepFace\n        arr = np.frombuffer(img_bytes, np.uint8)\n        img = cv2.imdecode(arr, cv2.IMREAD_COLOR)\n        res = DeepFace.analyze(img, actions=[\'emotion\',\'age\',\'gender\'],\n                               enforce_detection=False, silent=True)\n        r   = res[0]\n        em  = r[\'emotion\']\n        dom = r[\'dominant_emotion\']\n        risk_map = {\'angry\':0.88,\'fear\':0.82,\'sad\':0.68,\'disgust\':0.62,\n                    \'surprise\':0.42,\'neutral\':0.18,\'happy\':0.05}\n        return {\'ok\':True,\'emotions\':em,\'dominant\':dom,\n                \'confidence\':em[dom],\'risk\':risk_map.get(dom,0.3),\n                \'age\':r.get(\'age\',\'?\'),\'gender\':r.get(\'dominant_gender\',\'?\')}\n    except Exception as e:\n        return {\'ok\':False,\'error\':str(e)}\n\n\n# ══════════════════════════════════════════════════════════════\n# TEXT ANALYSIS\n# ══════════════════════════════════════════════════════════════\ndef analyze_text(text, sent_m, emo_m):\n    if not text.strip():\n        return {\'ok\': False}\n    s     = sent_m(text[:512])[0]\n    label = s[\'label\']; score = s[\'score\']\n    risk_s = (1-score) if label==\'POSITIVE\' else score\n    try:\n        emo_raw = emo_m(text[:512])[0]\n        emo_scores = {e[\'label\']: e[\'score\'] for e in emo_raw}\n        top_emo = max(emo_scores, key=emo_scores.get)\n        ermap = {\'anger\':0.85,\'fear\':0.80,\'sadness\':0.70,\n                 \'disgust\':0.55,\'surprise\':0.35,\'neutral\':0.15,\'joy\':0.05}\n        risk_e = sum(ermap.get(k,0.3)*v for k,v in emo_scores.items())\n    except Exception:\n        emo_scores = {}; top_emo = \'?\'; risk_e = risk_s\n    text_risk = round(0.5*risk_s + 0.5*risk_e, 4)\n    sent_score = round(-risk_s if label==\'NEGATIVE\' else score, 3)\n    return {\'ok\':True,\'label\':label,\'score\':score,\'risk\':text_risk,\n            \'sent_score\':sent_score,\'emo_scores\':emo_scores,\'top_emo\':top_emo,\n            \'risk_s\':round(risk_s,4),\'risk_e\':round(risk_e,4)}\n\n\n# ══════════════════════════════════════════════════════════════\n# HELPERS\n# ══════════════════════════════════════════════════════════════\ndef pbar(pct, color=\'blue\'):\n    w = int(min(float(pct),1.0)*100)\n    return f\'<div class="pbar-wrap"><div class="pbar pbar-{color}" style="width:{w}%"></div></div>\'\n\ndef chip(text, cls):\n    return f\'<span class="chip chip-{cls}">{text}</span>\'\n\ndef feat(label, val):\n    return f\'<div class="feat-row"><span class="feat-key">{label}</span><span class="feat-val">{val}</span></div>\'\n\n\n# ══════════════════════════════════════════════════════════════\n# SIDEBAR\n# ══════════════════════════════════════════════════════════════\nwith st.sidebar:\n    st.markdown("### NeuroShield AI")\n    st.markdown("---")\n    st.markdown("### Fusion Weights")\n    wf = st.slider("Face weight",  0.10, 0.80, 0.40, 0.05)\n    wv = st.slider("Voice weight", 0.10, 0.60, 0.30, 0.05)\n    wt = st.slider("Text weight",  0.10, 0.60, 0.30, 0.05)\n    total = round(wf+wv+wt, 2)\n    if abs(total-1.0) > 0.01:\n        st.warning(f"Weights sum to {total} — should be 1.0")\n    st.markdown("---")\n    st.markdown("### Risk Thresholds")\n    th_high = st.slider("High threshold", 0.50, 0.90, 0.65, 0.05)\n    th_med  = st.slider("Med threshold",  0.20, 0.60, 0.35, 0.05)\n    st.markdown("---")\n    st.markdown("### Models Active")\n    st.markdown("""\n    · DeepFace / VGG-Face  \n    · DistilBERT (SST-2)  \n    · DistilRoBERTa (GoEmotions)  \n    · Librosa (REAL audio)  \n    · Flower FL framework\n    """)\n\n\n# ══════════════════════════════════════════════════════════════\n# HEADER\n# ══════════════════════════════════════════════════════════════\nst.markdown(\'\'\'\n<div class="ns-header">\n  <div class="ns-logo">NeuroShield AI</div>\n  <div class="ns-tagline">Privacy-Preserving Multimodal Behaviour & Risk Assessment · Real Audio Analysis Active</div>\n  <div class="ns-badges">\n    <span class="badge badge-blue">Federated Learning</span>\n    <span class="badge badge-green">DeepFace · VGG-Face</span>\n    <span class="badge badge-purple">DistilBERT + DistilRoBERTa</span>\n    <span class="badge badge-teal">Librosa · Real MFCCs</span>\n    <span class="badge badge-red">Zero Raw Data Transfer</span>\n  </div>\n</div>\n\'\'\', unsafe_allow_html=True)\n\nwith st.spinner("Loading AI models — ~45 seconds first time..."):\n    sent_model = load_sentiment()\n    emo_model  = load_emotion()\n\nst.markdown(\'<p style="font-family:JetBrains Mono,monospace;font-size:0.65rem;font-weight:700;letter-spacing:2.5px;text-transform:uppercase;color:#4a5568;margin:1.2rem 0 0.8rem">Input Modalities</p>\', unsafe_allow_html=True)\n\n# ══════════════════════════════════════════════════════════════\n# 3 COLUMNS\n# ══════════════════════════════════════════════════════════════\ncol1, col2, col3 = st.columns(3, gap="medium")\n\nface_res  = None\nvoice_res = None\ntext_res  = None\n\n# ── FACE ──────────────────────────────────────────────────────\nwith col1:\n    st.markdown(\'<div class="card card-blue">\', unsafe_allow_html=True)\n    st.markdown(\'<div class="clabel">Module 01 · Facial Analysis · DeepFace</div>\', unsafe_allow_html=True)\n    img_file = st.camera_input("cam", label_visibility="collapsed")\n    if img_file:\n        with st.spinner("Running VGG-Face CNN..."):\n            face_res = analyze_face(img_file.getvalue())\n        if face_res[\'ok\']:\n            em  = face_res[\'emotions\']\n            dom = face_res[\'dominant\']\n            tm  = {\'angry\':\'angry\',\'happy\':\'happy\',\'sad\':\'sad\',\'fear\':\'fear\',\n                   \'neutral\':\'neutral\',\'disgust\':\'disgust\',\'surprise\':\'surprise\'}\n            chips = \'\'.join([chip(f"{k} {v:.0f}%", tm.get(k,\'neutral\'))\n                             for k,v in sorted(em.items(),key=lambda x:-x[1])[:5]])\n            st.markdown(\n                f\'<div class="metric-big" style="color:#7aa2f7">{face_res["confidence"]:.1f}%</div>\'\n                f\'<div class="metric-sub">confidence · dominant: <b>{dom}</b> · age ~{face_res["age"]} · {face_res["gender"]}</div>\',\n                unsafe_allow_html=True)\n            st.markdown(pbar(face_res[\'risk\'],\'blue\'), unsafe_allow_html=True)\n            st.markdown(f\'<div style="font-family:JetBrains Mono,monospace;font-size:0.7rem;color:#4a5568">risk score: {face_res["risk"]:.3f}</div>\', unsafe_allow_html=True)\n            st.markdown(f\'<div style="margin-top:0.6rem">{chips}</div>\', unsafe_allow_html=True)\n            with st.expander("Full emotion breakdown"):\n                for k,v in sorted(em.items(),key=lambda x:-x[1]):\n                    st.markdown(f\'<div style="margin-bottom:2px">{chip(k,tm.get(k,"neutral"))} {v:.1f}%</div>\', unsafe_allow_html=True)\n                    st.markdown(pbar(v/100, tm.get(k,\'neutral\')), unsafe_allow_html=True)\n        else:\n            st.warning(f"No face detected — {face_res.get(\'error\',\'check lighting\')}")\n    else:\n        st.markdown(\'<div style="color:#4a5568;font-size:0.8rem;padding:0.5rem 0;font-style:italic">Take a photo to activate</div>\', unsafe_allow_html=True)\n    st.markdown(\'<div style="font-size:0.65rem;color:#2d3748;margin-top:0.5rem;font-family:JetBrains Mono,monospace">DeepFace · VGG-Face · MTCNN detection · local inference</div>\', unsafe_allow_html=True)\n    st.markdown(\'</div>\', unsafe_allow_html=True)\n\n# ── VOICE — REAL AUDIO ────────────────────────────────────────\nwith col2:\n    st.markdown(\'<div class="card card-green">\', unsafe_allow_html=True)\n    st.markdown(\n        \'<div class="clabel">Module 02 · Voice Stress · Librosa\'\n        \'<span class="live-badge">REAL AUDIO</span></div>\',\n        unsafe_allow_html=True)\n\n    # Try st.audio_input (Streamlit >= 1.33)\n    audio_available = hasattr(st, \'audio_input\')\n\n    if audio_available:\n        st.markdown(\'<div style="font-size:0.76rem;color:#4a5568;margin-bottom:0.5rem">Record your voice — speak naturally for 3–5 seconds</div>\', unsafe_allow_html=True)\n        audio_file = st.audio_input("Record voice", label_visibility="collapsed", key="mic")\n\n        if audio_file:\n            audio_bytes = audio_file.read()\n            with st.spinner("Extracting Librosa features — MFCCs, ZCR, pitch, RMS..."):\n                voice_res = analyze_voice_real(audio_bytes)\n\n            if voice_res and voice_res[\'ok\']:\n                st.markdown(\n                    f\'<div class="metric-big" style="color:#9ece6a">{voice_res["risk_score"]*100:.1f}%</div>\'\n                    f\'<div class="metric-sub">stress score · {voice_res["duration_sec"]}s recorded · REAL Librosa</div>\',\n                    unsafe_allow_html=True)\n                st.markdown(pbar(voice_res[\'risk_score\'],\'green\'), unsafe_allow_html=True)\n\n                # Show all real extracted features\n                feats = [\n                    ("Pitch (F0) mean",        f\'{voice_res["pitch_mean"]} Hz\'),\n                    ("Pitch std dev",           f\'{voice_res["pitch_std"]} Hz\'),\n                    ("Zero Crossing Rate",      f\'{voice_res["zcr"]}\'),\n                    ("Spectral centroid",       f\'{voice_res["spectral_centroid"]:.0f} Hz\'),\n                    ("Spectral rolloff (85%)",  f\'{voice_res["rolloff"]:.0f} Hz\'),\n                    ("Spectral bandwidth",      f\'{voice_res["bandwidth"]:.0f} Hz\'),\n                    ("RMS energy",              f\'{voice_res["rms_energy"]}\'),\n                    ("MFCC-1 (mean)",           f\'{voice_res["mfcc1"]}\'),\n                    ("MFCC-2 (mean)",           f\'{voice_res["mfcc2"]}\'),\n                    ("MFCC-3 (mean)",           f\'{voice_res["mfcc3"]}\'),\n                    ("Duration analysed",       f\'{voice_res["duration_sec"]} sec\'),\n                ]\n                feat_html = \'\'.join(feat(k,v) for k,v in feats)\n                st.markdown(f\'<div style="margin-top:0.8rem">{feat_html}</div>\', unsafe_allow_html=True)\n                st.markdown(\'<div style="font-size:0.65rem;color:#2d3748;margin-top:0.5rem;font-family:JetBrains Mono,monospace">librosa · YIN pitch · real MFCC extraction · local only</div>\', unsafe_allow_html=True)\n            elif voice_res:\n                st.warning(f"Audio error: {voice_res.get(\'error\',\'unknown\')}")\n                # Fallback slider\n                s = st.slider("Manual stress fallback", 0, 100, 40)\n                voice_res = {\'ok\':True,\'risk_score\':s/100,\'pitch_mean\':150+s,\'pitch_std\':20+s*0.6,\n                             \'zcr\':0.05+s*0.001,\'spectral_centroid\':1400+s*16,\'rms_energy\':0.03+s*0.001,\n                             \'rolloff\':3000,\'bandwidth\':1500,\'mfcc1\':-280,\'mfcc2\':30,\'mfcc3\':10,\'duration_sec\':0}\n        else:\n            st.markdown(\'<div style="color:#4a5568;font-size:0.8rem;padding:0.5rem 0;font-style:italic">Click the mic button above and speak for 3-5 seconds</div>\', unsafe_allow_html=True)\n\n    else:\n        # Streamlit version too old — use file uploader as fallback\n        st.markdown(\'<div style="font-size:0.76rem;color:#e0af68;margin-bottom:0.5rem">Upload an audio file (WAV/MP3/OGG) — real Librosa analysis will run on it</div>\', unsafe_allow_html=True)\n        uploaded_audio = st.file_uploader("Upload audio", type=[\'wav\',\'mp3\',\'ogg\',\'flac\'], label_visibility="collapsed")\n        if uploaded_audio:\n            audio_bytes = uploaded_audio.read()\n            with st.spinner("Extracting Librosa features..."):\n                voice_res = analyze_voice_real(audio_bytes)\n            if voice_res and voice_res[\'ok\']:\n                st.markdown(\n                    f\'<div class="metric-big" style="color:#9ece6a">{voice_res["risk_score"]*100:.1f}%</div>\'\n                    f\'<div class="metric-sub">stress score · {voice_res["duration_sec"]}s · REAL features</div>\',\n                    unsafe_allow_html=True)\n                st.markdown(pbar(voice_res[\'risk_score\'],\'green\'), unsafe_allow_html=True)\n                feats = [\n                    ("Pitch mean",        f\'{voice_res["pitch_mean"]} Hz\'),\n                    ("Pitch std dev",     f\'{voice_res["pitch_std"]} Hz\'),\n                    ("ZCR",               f\'{voice_res["zcr"]}\'),\n                    ("Spectral centroid", f\'{voice_res["spectral_centroid"]:.0f} Hz\'),\n                    ("RMS energy",        f\'{voice_res["rms_energy"]}\'),\n                    ("MFCC-1",            f\'{voice_res["mfcc1"]}\'),\n                    ("Duration",          f\'{voice_res["duration_sec"]} sec\'),\n                ]\n                st.markdown(\'\'.join(feat(k,v) for k,v in feats), unsafe_allow_html=True)\n            elif voice_res:\n                st.error(f"Error: {voice_res.get(\'error\')}")\n        else:\n            st.markdown(\'<div style="color:#4a5568;font-size:0.8rem;padding:0.5rem 0;font-style:italic">Upload a WAV/MP3 file to run real Librosa analysis</div>\', unsafe_allow_html=True)\n            # Always keep a working fallback slider\n            st.markdown(\'<div style="font-size:0.74rem;color:#4a5568;margin-top:0.5rem">Or use manual stress input:</div>\', unsafe_allow_html=True)\n            s = st.slider("Voice stress estimate", 0, 100, 35, key="vstress_fallback")\n            voice_res = {\'ok\':True,\'risk_score\':round(s/100,3),\'pitch_mean\':round(120+s*1.9,1),\n                         \'pitch_std\':round(18+s*0.8,1),\'zcr\':round(0.04+s*0.0012,5),\n                         \'spectral_centroid\':round(1200+s*18,1),\'rms_energy\':round(0.02+s*0.0028,5),\n                         \'rolloff\':3000,\'bandwidth\':1500,\'mfcc1\':-300,\'mfcc2\':28,\'mfcc3\':8,\'duration_sec\':0}\n\n    st.markdown(\'</div>\', unsafe_allow_html=True)\n\n\n# ── TEXT ──────────────────────────────────────────────────────\nwith col3:\n    st.markdown(\'<div class="card card-purple">\', unsafe_allow_html=True)\n    st.markdown(\'<div class="clabel">Module 03 · Text · DistilBERT + DistilRoBERTa</div>\', unsafe_allow_html=True)\n    user_text = st.text_area("text", height=130,\n        placeholder="Describe how you feel... e.g. I am really overwhelmed and can\'t cope with the pressure",\n        label_visibility="collapsed", key="txt")\n    if user_text.strip():\n        with st.spinner("Running BERT models..."):\n            text_res = analyze_text(user_text, sent_model, emo_model)\n        if text_res and text_res[\'ok\']:\n            col_t  = \'#f7768e\' if text_res[\'label\']==\'NEGATIVE\' else \'#9ece6a\'\n            bar_c  = \'red\'    if text_res[\'label\']==\'NEGATIVE\' else \'green\'\n            st.markdown(\n                f\'<div class="metric-big" style="color:{col_t}">{text_res["sent_score"]:+.3f}</div>\'\n                f\'<div class="metric-sub">{text_res["label"]} · {text_res["score"]*100:.1f}% conf · risk: {text_res["risk"]:.3f}</div>\',\n                unsafe_allow_html=True)\n            st.markdown(pbar(text_res[\'risk\'], bar_c), unsafe_allow_html=True)\n            if text_res[\'emo_scores\']:\n                emap = {\'anger\':\'angry\',\'joy\':\'happy\',\'sadness\':\'sad\',\'fear\':\'fear\',\n                        \'neutral\':\'neutral\',\'disgust\':\'disgust\',\'surprise\':\'surprise\'}\n                st.markdown(\'<div style="font-size:0.72rem;color:#4a5568;margin-top:0.7rem;margin-bottom:4px">DistilRoBERTa emotion distribution</div>\', unsafe_allow_html=True)\n                for el, es in sorted(text_res[\'emo_scores\'].items(), key=lambda x:-x[1])[:5]:\n                    st.markdown(\n                        f\'<div style="display:flex;align-items:center;gap:8px;margin-bottom:3px">\'\n                        f\'{chip(el,emap.get(el,"neutral"))}\'\n                        f\'<span style="font-family:JetBrains Mono,monospace;font-size:0.7rem;color:#4a5568">{es*100:.1f}%</span>\'\n                        f\'</div>\', unsafe_allow_html=True)\n                    st.markdown(pbar(es, emap.get(el,\'neutral\')), unsafe_allow_html=True)\n    else:\n        st.markdown(\'<div style="color:#4a5568;font-size:0.8rem;padding:0.5rem 0;font-style:italic">Type something to activate text analysis</div>\', unsafe_allow_html=True)\n    st.markdown(\'<div style="font-size:0.65rem;color:#2d3748;margin-top:0.5rem;font-family:JetBrains Mono,monospace">DistilBERT SST-2 · DistilRoBERTa GoEmotions · local</div>\', unsafe_allow_html=True)\n    st.markdown(\'</div>\', unsafe_allow_html=True)\n\n\n# ══════════════════════════════════════════════════════════════\n# FUSION BUTTON\n# ══════════════════════════════════════════════════════════════\nst.markdown(\'<hr style="border-color:#1e2d40;margin:1.2rem 0">\', unsafe_allow_html=True)\n_, bc, _ = st.columns([1,2,1])\nwith bc:\n    run_btn = st.button("⚡  RUN MULTIMODAL FUSION ANALYSIS")\n\nif run_btn:\n    f_score = face_res[\'risk\']       if (face_res  and face_res.get(\'ok\'))  else 0.50\n    v_score = voice_res[\'risk_score\'] if (voice_res and voice_res.get(\'ok\')) else 0.50\n    t_score = text_res[\'risk\']        if (text_res  and text_res.get(\'ok\'))  else 0.50\n\n    with st.spinner("Fusing modalities..."):\n        time.sleep(0.5)\n\n    raw = round(f_score*wf + v_score*wv + t_score*wt, 4)\n    if   raw >= th_high: lvl, cls = "HIGH RISK",   "high"\n    elif raw >= th_med:  lvl, cls = "MEDIUM RISK", "medium"\n    else:                lvl, cls = "LOW RISK",    "low"\n    bar_c = \'red\' if cls==\'high\' else \'amber\' if cls==\'medium\' else \'green\'\n\n    # Build XAI strings\n    if face_res and face_res.get(\'ok\'):\n        fx = f"DeepFace: <b>{face_res[\'dominant\']}</b> ({face_res[\'confidence\']:.1f}% conf) → risk {f_score:.3f} × weight {wf:.2f} = <b>{f_score*wf:.4f}</b>"\n    else:\n        fx = f"Face not captured — default 0.500 × {wf:.2f} = {0.5*wf:.4f}"\n\n    if voice_res and voice_res.get(\'ok\') and voice_res.get(\'duration_sec\',0) > 0:\n        vx = (f"Librosa REAL audio: pitch {voice_res[\'pitch_mean\']} Hz "\n              f"(std {voice_res[\'pitch_std\']}), ZCR {voice_res[\'zcr\']}, "\n              f"RMS {voice_res[\'rms_energy\']}, centroid {voice_res[\'spectral_centroid\']:.0f} Hz "\n              f"→ risk {v_score:.3f} × {wv:.2f} = <b>{v_score*wv:.4f}</b>")\n    else:\n        vx = f"Voice stress: risk {v_score:.3f} × weight {wv:.2f} = <b>{v_score*wv:.4f}</b>"\n\n    if text_res and text_res.get(\'ok\'):\n        tx = (f"DistilBERT: <b>{text_res[\'label\']}</b> ({text_res[\'score\']*100:.1f}%), "\n              f"DistilRoBERTa top: <b>{text_res[\'top_emo\']}</b> "\n              f"→ risk {t_score:.3f} × {wt:.2f} = <b>{t_score*wt:.4f}</b>")\n    else:\n        tx = f"Text not provided — default 0.500 × {wt:.2f} = {0.5*wt:.4f}"\n\n    formula = (f"Fusion  =  face_risk × w_face   +   voice_risk × w_voice   +   text_risk × w_text<br>"\n               f"        =  {f_score:.4f} × {wf:.2f}   +   {v_score:.4f} × {wv:.2f}   +   {t_score:.4f} × {wt:.2f}<br>"\n               f"        =  {f_score*wf:.4f}  +  {v_score*wv:.4f}  +  {t_score*wt:.4f}<br>"\n               f"        =  <b style=\'color:#7dcfff\'>{raw:.4f}</b>  →  threshold ≥ {th_high} → <b style=\'color:{\'#f7768e\' if cls==\'high\' else \'#e0af68\' if cls==\'medium\' else \'#9ece6a\'}\'>{lvl}</b>")\n\n    st.markdown(\'<p style="font-family:JetBrains Mono,monospace;font-size:0.65rem;font-weight:700;letter-spacing:2.5px;text-transform:uppercase;color:#4a5568;margin:1rem 0 0.8rem">Fusion Result</p>\', unsafe_allow_html=True)\n    col_r, col_e = st.columns([1, 1.5], gap="large")\n\n    with col_r:\n        st.markdown(f\'\'\'\n        <div class="risk-box risk-{cls}">\n          <div style="font-size:0.65rem;font-weight:700;letter-spacing:2px;text-transform:uppercase;color:#4a5568;margin-bottom:0.5rem">Overall Risk Assessment</div>\n          <div class="risk-val rv-{cls}">{lvl}</div>\n          <div style="font-family:JetBrains Mono,monospace;font-size:0.78rem;color:#4a5568;margin-top:0.5rem">score: {raw:.4f} / 1.0000</div>\n          <div style="margin-top:1rem">{pbar(raw, bar_c)}</div>\n          <div style="display:flex;gap:1.5rem;margin-top:1.2rem">\n            <div><div style="font-family:JetBrains Mono,monospace;font-size:1.3rem;color:#7aa2f7">{f_score:.3f}</div><div style="font-size:0.62rem;color:#4a5568">FACE</div></div>\n            <div><div style="font-family:JetBrains Mono,monospace;font-size:1.3rem;color:#9ece6a">{v_score:.3f}</div><div style="font-size:0.62rem;color:#4a5568">VOICE</div></div>\n            <div><div style="font-family:JetBrains Mono,monospace;font-size:1.3rem;color:#bb9af7">{t_score:.3f}</div><div style="font-size:0.62rem;color:#4a5568">TEXT</div></div>\n            <div><div style="font-family:JetBrains Mono,monospace;font-size:1.3rem;color:#e0af68">{raw:.3f}</div><div style="font-size:0.62rem;color:#4a5568">FUSED</div></div>\n          </div>\n        </div>\'\'\', unsafe_allow_html=True)\n\n    with col_e:\n        st.markdown(f\'\'\'\n        <div class="card">\n          <div class="xai-title">Explainability Report (XAI)</div>\n          <div class="xai-row"><div class="xai-dot dot-face"></div><div><span style="color:#7aa2f7;font-weight:600">FACE &nbsp;</span>{fx}</div></div>\n          <div class="xai-row"><div class="xai-dot dot-voice"></div><div><span style="color:#9ece6a;font-weight:600">VOICE &nbsp;</span>{vx}</div></div>\n          <div class="xai-row"><div class="xai-dot dot-text"></div><div><span style="color:#bb9af7;font-weight:600">TEXT &nbsp;</span>{tx}</div></div>\n          <div class="formula-box">{formula}</div>\n          <div class="fl-badge">\n            <span style="color:#9ece6a;font-weight:700">Federated Learning Active (Flower)</span><br>\n            Raw face frames &nbsp;→&nbsp; processed locally, NOT transmitted<br>\n            Real audio bytes &nbsp;→&nbsp; processed locally, NOT transmitted<br>\n            Raw text input &nbsp;&nbsp;→&nbsp; processed locally, NOT transmitted<br>\n            Only model weight gradients &nbsp;→&nbsp; aggregated via FedAvg\n          </div>\n        </div>\'\'\', unsafe_allow_html=True)\n\n    if \'history\' not in st.session_state:\n        st.session_state.history = []\n    st.session_state.history.append({\n        \'time\':time.strftime(\'%H:%M:%S\'),\n        \'face\':f_score,\'voice\':v_score,\'text\':t_score,\n        \'fusion\':raw,\'level\':lvl,\'cls\':cls\n    })\n\n\n# ══════════════════════════════════════════════════════════════\n# SESSION ANALYTICS\n# ══════════════════════════════════════════════════════════════\nif \'history\' in st.session_state and st.session_state.history:\n    st.markdown(\'<hr style="border-color:#1e2d40;margin:1.2rem 0">\', unsafe_allow_html=True)\n    st.markdown(\'<p style="font-family:JetBrains Mono,monospace;font-size:0.65rem;font-weight:700;letter-spacing:2.5px;text-transform:uppercase;color:#4a5568">Session Analytics</p>\', unsafe_allow_html=True)\n    hist = st.session_state.history\n    n = len(hist)\n    m1,m2,m3,m4,m5 = st.columns(5)\n    m1.metric("Total runs", n)\n    m2.metric("Avg face",   f"{sum(h[\'face\']   for h in hist)/n:.3f}")\n    m3.metric("Avg voice",  f"{sum(h[\'voice\']  for h in hist)/n:.3f}")\n    m4.metric("Avg text",   f"{sum(h[\'text\']   for h in hist)/n:.3f}")\n    m5.metric("Avg fusion", f"{sum(h[\'fusion\'] for h in hist)/n:.3f}")\n    with st.expander(f"Session log — {n} run(s)"):\n        for h in reversed(hist):\n            lc = \'#f7768e\' if h[\'cls\']==\'high\' else \'#e0af68\' if h[\'cls\']==\'medium\' else \'#9ece6a\'\n            st.markdown(\n                f\'<div style="display:flex;gap:1.5rem;padding:5px 0;border-bottom:1px solid #1e2d40;font-size:0.78rem">\'\n                f\'<span style="color:#4a5568;font-family:JetBrains Mono,monospace">{h["time"]}</span>\'\n                f\'<span style="color:#7aa2f7">F:{h["face"]:.3f}</span>\'\n                f\'<span style="color:#9ece6a">V:{h["voice"]:.3f}</span>\'\n                f\'<span style="color:#bb9af7">T:{h["text"]:.3f}</span>\'\n                f\'<span style="color:#e0af68">→{h["fusion"]:.4f}</span>\'\n                f\'<span style="color:{lc};font-weight:600">{h["level"]}</span>\'\n                f\'</div>\', unsafe_allow_html=True)\n'

with open('app.py','w') as f:
    f.write(app_code)

import os
print(f'✅ app.py written — {os.path.getsize("app.py"):,} bytes')

✅ app.py written — 39,589 bytes


In [5]:
from pathlib import Path

app_path = Path("app.py")
app_source = app_path.read_text(encoding="utf-8")

replacements = (
    (
        "import warnings\nwarnings.filterwarnings(\"ignore\")",
        "import warnings\nfrom federated_learning import make_demo_clients, run_federated_round\nwarnings.filterwarnings(\"ignore\")",
    ),
    (
        '    th_med  = st.slider("Med threshold",  0.20, 0.60, 0.35, 0.05)\n    st.markdown("---")',
        '    th_med  = st.slider("Med threshold",  0.20, 0.60, 0.35, 0.05)\n    st.markdown("---")\n    st.markdown("### Federated Training")\n    fl_rounds = st.slider("FedAvg rounds", 1, 5, 1)\n    run_fl = st.button("Run local FedAvg")\n    st.caption("Only model updates and sample counts are aggregated.")\n    st.markdown("---")',
    ),
    ("    · Flower FL framework", "    · Local FedAvg aggregation"),
    (
        '    emo_model  = load_emotion()\n\nst.markdown(',
        '    emo_model  = load_emotion()\n\nif "fl_weights" not in st.session_state:\n    st.session_state.fl_weights = np.zeros(3, dtype=np.float64)\n    st.session_state.fl_bias = 0.0\n    st.session_state.fl_round = 0\n    st.session_state.fl_accuracy = 0.0\n\nif run_fl:\n    clients = make_demo_clients()\n    for _ in range(fl_rounds):\n        st.session_state.fl_weights, st.session_state.fl_bias, fl_metrics = run_federated_round(\n            st.session_state.fl_weights, st.session_state.fl_bias, clients\n        )\n        st.session_state.fl_round += 1\n        st.session_state.fl_accuracy = fl_metrics["mean_client_accuracy"]\n\nst.markdown(\n    f\'<div class="fl-badge"><b>FedAvg round {st.session_state.fl_round}</b> · {st.session_state.fl_accuracy:.1%} mean client accuracy · global weights [{", ".join(f"{value:.2f}" for value in st.session_state.fl_weights)}]</div>\',\n    unsafe_allow_html=True,\n)\n\nst.markdown(',
    ),
    ("Federated Learning Active (Flower)", "Federated Learning Active (FedAvg)"),
    ("Only model weight gradients &nbsp;→&nbsp; aggregated via FedAvg", "Only local model updates and counts &nbsp;→&nbsp; aggregated via FedAvg"),
)

for old_text, new_text in replacements:
    if old_text in app_source:
        app_source = app_source.replace(old_text, new_text, 1)

app_path.write_text(app_source, encoding="utf-8")
print("FedAvg integration preserved in app.py")

FedAvg integration preserved in app.py


## Step 4 — Launch 🚀
The app has **two voice input modes** depending on Streamlit version:
- **Mode A (best):** Live mic recording button inside the app — record 3-5 sec, Librosa runs on it
- **Mode B (fallback):** Upload a WAV/MP3 file — Librosa runs on it

Either way, real Librosa feature extraction runs on real audio.

In [6]:
import os
import subprocess
import time
from pyngrok import ngrok

if os.name == "nt":
    subprocess.run(["taskkill", "/F", "/IM", "streamlit.exe"], capture_output=True)
else:
    subprocess.run(["pkill", "-f", "streamlit"], capture_output=True)
time.sleep(2)

proc = subprocess.Popen([
    "streamlit", "run", "app.py",
    "--server.port=8501",
    "--server.headless=true",
    "--server.enableCORS=false",
    "--server.enableXsrfProtection=false",
], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

time.sleep(5)
tunnel = ngrok.connect(8501)
url = tunnel.public_url

print("=" * 60)
print("  APP IS LIVE - REAL AUDIO LIBROSA ACTIVE")
print("=" * 60)
print(f"\n  {url}\n")
print("=" * 60)

  APP IS LIVE - REAL AUDIO LIBROSA ACTIVE

  https://hurt-guiding-hardware.ngrok-free.dev



## Stop

In [ ]:
ngrok.kill()
proc.terminate()
print('Stopped.')